<a href="https://colab.research.google.com/github/stella1298/AIFFEL_Quest_EPA/blob/master/LLM/LLM06/261002_MCP_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
#Step 0 : 환경 준비와 LLM 백엔드 추상화

# 메이저 버전에 상한을 둡니다. mcp는 2.0에서 서버 클래스 이름이 바뀌었고 이 레슨은 2.x를
# 기준으로 쓰여 있으므로, 상한이 없으면 3.0이 나오는 날 임포트 단계에서 멈춥니다
!pip install -q "transformers==4.57.*" "accelerate==1.14.*" "bitsandbytes==0.50.*"
!pip install -q "langchain-huggingface==0.3.*" "langchain-text-splitters==0.3.*" \
                "sentence-transformers==5.7.*"
!pip install -q "mcp==2.*" "openai==2.*"

# 설치가 끝난 뒤 pip이 의존성 충돌을 보고하면 [런타임] - [세션 다시 시작]을 한 번 누르고
# 이 셀 아래부터 다시 실행하세요. 버전을 고정하면 실습 환경의 기본 패키지와 부딪히는 경우가 있습니다

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 134.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 64.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 94.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.27.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
import torch, os, json, time, re, asyncio
from importlib.metadata import version, PackageNotFoundError

print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


def pkgver(dist):
    """배포판 이름으로 설치된 버전을 읽습니다.

    모듈의 __version__에 의존하지 않는 이유가 있습니다. 그 속성을 노출하지 않는
    패키지가 있어서(프라이빗 RAG 노드의 qdrant-client가 그렇습니다), 설치 메타데이터를
    읽는 쪽이 어떤 패키지에나 통합니다.
    """
    try:
        return version(dist)
    except PackageNotFoundError:
        return "미설치"


# 설치된 버전을 기록해둡니다. 문제가 생기면 이 숫자부터 확인합니다
print("\n[설치된 버전]")
for d in ["transformers", "mcp", "langchain-huggingface", "langchain-text-splitters",
          "bitsandbytes", "accelerate", "openai"]:
    print(f"  {d:26s} {pkgver(d)}")

GPU : Tesla T4
VRAM : 15.6 GB

[설치된 버전]
  transformers               4.57.6
  mcp                        2.2.0
  langchain-huggingface      0.3.1
  langchain-text-splitters   0.3.11
  bitsandbytes               0.50.2
  accelerate                 1.14.0
  openai                     2.54.0


In [4]:
# 백엔드 선택. 로컬 GPU로 실습하려면 "local", 원격 엔드포인트를 쓰려면 "endpoint"
BACKEND = "local"

# endpoint 사용 시 채울 값 (강사가 안내하는 주소를 입력)
ENDPOINT_URL = "http://localhost:8000/v1"
ENDPOINT_MODEL = "Qwen/Qwen2.5-7B-Instruct"
ENDPOINT_KEY = "dummy-key"        # 사내 서버는 인증이 없는 경우가 많습니다

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

In [6]:
if BACKEND == "local":
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    _model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config, device_map="auto",
        dtype=torch.float16,      # 양자화되지 않는 계층의 정밀도. T4는 bf16 미지원이라 명시합니다
    )
    _model.eval()
    print(f"로컬 모델 로드 완료. VRAM {torch.cuda.memory_allocated()/1e9:.2f} GB")
else:
    from openai import OpenAI
    _client = OpenAI(base_url=ENDPOINT_URL, api_key=ENDPOINT_KEY)
    print("엔드포인트 클라이언트 준비 완료")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

로컬 모델 로드 완료. VRAM 8.93 GB


In [7]:
def chat(messages, tools=None, max_new_tokens=512, temperature=0.2):
    """백엔드에 무관하게 동작하는 대화 함수.
    반환값 : {"content": str, "tool_calls": [{"name":..., "arguments": {...}}]}
    """
    do_sample = temperature > 0

    if BACKEND == "endpoint":
        kwargs = dict(model=ENDPOINT_MODEL, messages=messages, max_tokens=max_new_tokens)
        if do_sample:
            kwargs["temperature"] = temperature
        if tools:
            kwargs["tools"] = [{"type": "function", "function": t} for t in tools]
        r = _client.chat.completions.create(**kwargs)
        msg = r.choices[0].message
        calls = []
        for tc in (msg.tool_calls or []):
            calls.append({"name": tc.function.name,
                          "arguments": json.loads(tc.function.arguments)})
        return {"content": msg.content or "", "tool_calls": calls}

    # local : chat template에 tools를 넘기면 모델이 도구 호출 형식으로 응답합니다
    text = tokenizer.apply_chat_template(
        messages, tools=tools, add_generation_prompt=True, tokenize=False
    )
    ids = tokenizer(text, return_tensors="pt").to(_model.device)
    gen = dict(max_new_tokens=max_new_tokens, do_sample=do_sample,
               pad_token_id=tokenizer.eos_token_id)
    if do_sample:
        # 그리디 디코딩에서는 temperature를 아예 넘기지 않습니다. 넘기면 transformers가
        # 호출마다 "샘플링 전용 인자"라는 경고를 찍어 측정 로그가 읽기 어려워집니다
        gen["temperature"] = temperature
    with torch.no_grad():
        out = _model.generate(**ids, **gen)
    raw = tokenizer.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)
    return parse_local_output(raw)

In [8]:
def parse_local_output(raw: str):
    """Qwen 계열은 <tool_call>{...}</tool_call> 형태로 도구 호출을 출력합니다"""
    calls = []
    # 닫는 태그 앞의 JSON 을 찾습니다. 7B 모델은 여는 태그 <tool_call> 을 빠뜨리는 일이 잦아서
    # 여는 태그가 없어도 호출로 인정합니다
    for m in re.finditer(r"(\{.*?\})\s*</tool_call>", raw, re.S):
        try:
            d = json.loads(m.group(1))
            if "name" in d:
                calls.append({"name": d.get("name"), "arguments": d.get("arguments", {})})
        except json.JSONDecodeError:
            pass                      # 형식이 깨진 호출. 뒤에서 실패로 다루게 됩니다
    content = re.sub(r"(<tool_call>)?\s*\{.*?\}\s*</tool_call>", "", raw, flags=re.S).strip()
    if not calls and "tool_call>" in raw:
        # 태그 조각은 있는데 읽을 수 있는 호출이 하나도 없는 경우입니다
        content = "[도구 호출 형식이 깨졌습니다] " + content
    return {"content": content, "tool_calls": calls}

In [9]:
# 동작 확인
r = chat([{"role": "user", "content": "한 문장으로 자기소개 해주세요."}], max_new_tokens=80)
print(r["content"])

안녕하세요, 저는 Qwen이라고 합니다.阿里巴巴云开发的大型语言模型，很高兴为您服务！


In [10]:
def assistant_tool_msg(content, calls):
    """도구 호출 이력을 대화 메시지로 만듭니다. 백엔드에 따라 인자 형식이 다릅니다"""
    def enc(args):
        # local : chat template이 tojson을 적용하므로 dict 그대로
        # endpoint : OpenAI 규격이 문자열을 요구
        return json.dumps(args, ensure_ascii=False) if BACKEND == "endpoint" else args

    return {"role": "assistant", "content": content,
            "tool_calls": [{"type": "function",
                            "function": {"name": c["name"], "arguments": enc(c["arguments"])}}
                           for c in calls]}


def safe_call(name, args):
    """도구를 실행하고, 실패하면 모델이 읽을 수 있는 문자열로 되돌려줍니다"""
    fn = TOOL_IMPL.get(name)
    if fn is None:
        return f"알 수 없는 도구입니다: {name}. 사용 가능한 도구만 호출하세요."
    try:
        return str(fn(**args))
    except TypeError as e:
        return f"인자가 올바르지 않습니다: {e}. 도구 스키마의 인자 이름을 확인해 다시 호출하세요."
    except Exception as e:
        return f"도구 실행 중 오류: {type(e).__name__}: {e}"

In [11]:
import pandas as pd
import numpy as np

# 실습용 사내 매출 데이터
np.random.seed(42)
sales = pd.DataFrame({
    "지역": ["서울", "경기", "부산", "대구", "광주"] * 12,
    "월": sorted(list(range(1, 13)) * 5),
    "매출": np.random.randint(800, 5000, 60),
    "고객수": np.random.randint(40, 300, 60),
})
sales.head()

,지역,월,매출,고객수
0,서울,1,1660,254
1,경기,1,4572,291
2,부산,1,3892,229
3,대구,1,1266,252
4,광주,1,4244,247


In [12]:
# 도구 구현부. 실제 파이썬 함수입니다
def query_sales(지역: str = None, 월: int = None) -> str:
    df = sales.copy()
    if 지역:
        df = df[df["지역"] == 지역]
    if 월:
        df = df[df["월"] == 월]
    if df.empty:
        return "조건에 맞는 데이터가 없습니다."
    return df.to_string(index=False)


def calculate(expression: str) -> str:
    try:
        # 실습용 단순 계산기입니다. 실무에서는 eval을 쓰지 않습니다
        allowed = set("0123456789+-*/(). ")
        if not set(expression) <= allowed:
            return "허용되지 않은 문자가 있습니다."
        return str(eval(expression))
    except Exception as e:
        return f"계산 실패 : {e}"


TOOL_IMPL = {"query_sales": query_sales, "calculate": calculate}

In [13]:
# 도구 스키마. 모델은 이 설명만 보고 언제 무엇을 부를지 판단합니다
TOOLS = [
    {
        "name": "query_sales",
        "description": "사내 매출 데이터를 조회합니다. 지역별 또는 월별 매출과 고객수가 필요할 때 사용하세요.",
        "parameters": {
            "type": "object",
            "properties": {
                "지역": {"type": "string", "description": "서울, 경기, 부산, 대구, 광주 중 하나"},
                "월": {"type": "integer", "description": "1에서 12 사이의 월"},
            },
        },
    },
    {
        "name": "calculate",
        "description": "산술식을 계산합니다. 조회한 숫자들의 합계, 평균, 비율을 구할 때 사용하세요.",
        "parameters": {
            "type": "object",
            "properties": {
                "expression": {"type": "string", "description": "예: (1200+3400)/2"},
            },
            "required": ["expression"],
        },
    },
]

In [14]:
AGENT_SYSTEM = """당신은 사내 데이터 분석 에이전트입니다.
사용 가능한 도구로 필요한 정보를 조회하고 계산한 뒤, 최종 답변을 한국어로 제시하세요.
도구가 더 필요하지 않으면 도구를 호출하지 말고 최종 답변만 작성하세요.
숫자 계산은 직접 하지 말고 calculate 도구를 사용하세요."""


def run_agent(question, max_steps=6, verbose=True):
    messages = [{"role": "system", "content": AGENT_SYSTEM},
                {"role": "user", "content": question}]
    trace = []

    for step in range(max_steps):
        r = chat(messages, tools=TOOLS)

        if not r["tool_calls"]:                       # 종료 판단은 모델이 합니다
            if verbose:
                print(f"[{step+1}] 최종 답변")
            return {"answer": r["content"], "trace": trace, "steps": step + 1}

        messages.append(assistant_tool_msg(r["content"], r["tool_calls"]))

        for c in r["tool_calls"]:
            result = safe_call(c["name"], c["arguments"])
            trace.append({"step": step + 1, "tool": c["name"], "args": c["arguments"]})
            if verbose:
                print(f"[{step+1}] {c['name']}({c['arguments']}) -> {str(result)[:60]}")
            messages.append({"role": "tool", "name": c["name"], "content": str(result)})

    return {"answer": "최대 단계에 도달했습니다.", "trace": trace, "steps": max_steps}

In [15]:
result = run_agent("서울의 3월 매출과 부산의 3월 매출을 합치면 얼마야?")
print("\n답변 :", result["answer"])

[1] query_sales({'지역': '서울', '월': 3}) -> 지역  월   매출  고객수
서울  3 3191   80
[1] query_sales({'지역': '부산', '월': 3}) -> 지역  월   매출  고객수
부산  3 1984   54
[2] 최종 답변

답변 : 서울과 부산의 3월 매출을 합하면 3191 + 1984 = 5175입니다.


In [16]:
# 실행마다 경로가 달라지는지 확인합니다
for i in range(3):
    r = run_agent("경기 지역에서 가장 매출이 높은 월은?", verbose=False)
    tools_used = [t["tool"] for t in r["trace"]]
    print(f"{i+1}회차 : {r['steps']}단계, 도구 {tools_used}")

1회차 : 2단계, 도구 ['query_sales']
2회차 : 2단계, 도구 ['query_sales']
3회차 : 2단계, 도구 ['query_sales']


In [17]:
# 나쁜 설명 : 기능만 있고 언제 쓸지가 없습니다
TOOLS_BAD = [
    {"name": "query_sales", "description": "데이터를 조회합니다.",
     "parameters": {"type": "object", "properties": {
         "지역": {"type": "string"}, "월": {"type": "integer"}}}},
    {"name": "calculate", "description": "계산합니다.",
     "parameters": {"type": "object", "properties": {
         "expression": {"type": "string"}}, "required": ["expression"]}},
]

# 평가 세트 : (질문, 첫 번째로 불러야 하는 도구)
# 앞 6개는 판단이 쉬운 질문, 뒤 6개는 의도적으로 애매하게 만든 질문입니다.
# 애매한 질문이 있어야 설명 품질의 차이가 숫자로 드러납니다
tool_eval = [
    ("서울 5월 매출 알려줘", "query_sales"),
    ("부산의 모든 월 고객수를 보여줘", "query_sales"),
    ("1200 더하기 3400은?", "calculate"),
    ("대구 7월 매출을 2로 나누면?", "query_sales"),
    ("광주 지역 데이터 전체", "query_sales"),
    ("(500+700)*3 계산해줘", "calculate"),
    # 이하 애매한 질문 : 숫자가 등장하지만 먼저 조회가 필요하거나, 반대로 조회가 불필요한 경우
    ("서울과 부산 중 어디가 더 팔렸어?", "query_sales"),
    ("객단가 좀 뽑아줘", "query_sales"),
    ("아까 본 1200과 3400의 평균", "calculate"),
    ("전체 평균 매출이 궁금해", "query_sales"),
    ("3개월치를 12개월로 환산하면 4배지?", "calculate"),
    ("경기 실적 추세 알려줘", "query_sales"),
]

In [18]:
def first_tool_accuracy(tools, dataset, temperature=0.0, show_miss=False):
    """첫 도구 선택 정확도.

    temperature=0.0으로 고정하는 이유가 있습니다. 우리가 재려는 것은 설명 품질의 효과인데,
    샘플링을 켜면 실행마다 값이 흔들려 그 효과와 우연을 구분할 수 없습니다.
    측정할 때는 그리디 디코딩으로 고정하고, 대신 질문 수를 늘려 해상도를 확보합니다.
    """
    hits, misses = 0, []
    for q, gold in dataset:
        r = chat([{"role": "system", "content": AGENT_SYSTEM},
                  {"role": "user", "content": q}], tools=tools, temperature=temperature)
        picked = r["tool_calls"][0]["name"] if r["tool_calls"] else "없음"
        if picked == gold:
            hits += 1
        else:
            misses.append((q, gold, picked))
    if show_miss:
        for q, gold, picked in misses:
            print(f"    X {q} (정답 {gold} / 선택 {picked})")
    return hits / len(dataset)


print(f"질문 {len(tool_eval)}개로 측정합니다 (해상도 {1/len(tool_eval):.1%})\n")

print("나쁜 설명")
acc_bad = first_tool_accuracy(TOOLS_BAD, tool_eval, show_miss=True)
print(f"  정확도 {acc_bad:.0%}\n")

print("좋은 설명")
acc_good = first_tool_accuracy(TOOLS, tool_eval, show_miss=True)
print(f"  정확도 {acc_good:.0%}")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


질문 12개로 측정합니다 (해상도 8.3%)

나쁜 설명
    X 객단가 좀 뽑아줘 (정답 query_sales / 선택 없음)
    X 전체 평균 매출이 궁금해 (정답 query_sales / 선택 없음)
  정확도 83%

좋은 설명
    X 객단가 좀 뽑아줘 (정답 query_sales / 선택 없음)
    X 전체 평균 매출이 궁금해 (정답 query_sales / 선택 없음)
    X 3개월치를 12개월로 환산하면 4배지? (정답 calculate / 선택 없음)
    X 경기 실적 추세 알려줘 (정답 query_sales / 선택 없음)
  정확도 67%


In [19]:
# 실습 : 아래 도구 설명을 개선해 정확도를 올려보세요
# 힌트 : 언제 써야 하는지, 언제 쓰지 말아야 하는지, 인자 예시를 넣습니다

TOOLS_MY = [
    {
        "name": "query_sales",
        "description": "여기에 개선한 설명을 작성하세요",
        "parameters": {"type": "object", "properties": {
            "지역": {"type": "string", "description": ""},
            "월": {"type": "integer", "description": ""}}},
    },
    {
        "name": "calculate",
        "description": "여기에 개선한 설명을 작성하세요",
        "parameters": {"type": "object", "properties": {
            "expression": {"type": "string", "description": ""}}, "required": ["expression"]},
    },
]

print("내 설명")
print(f"  정확도 {first_tool_accuracy(TOOLS_MY, tool_eval, show_miss=True):.0%}")

내 설명
    X 대구 7월 매출을 2로 나누면? (정답 query_sales / 선택 calculate)
    X 객단가 좀 뽑아줘 (정답 query_sales / 선택 없음)
    X 전체 평균 매출이 궁금해 (정답 query_sales / 선택 없음)
    X 3개월치를 12개월로 환산하면 4배지? (정답 calculate / 선택 없음)
    X 경기 실적 추세 알려줘 (정답 query_sales / 선택 없음)
  정확도 58%


In [20]:
TOOLS_MY = [
    {
        "name": "query_sales",
        "description": (
            "사내 매출 데이터베이스에서 실제 수치를 조회합니다. "
            "지역별 또는 월별 매출액과 고객수를 알아야 할 때 가장 먼저 사용하세요. "
            "숫자를 이미 알고 있어 계산만 필요한 경우에는 사용하지 마세요."
        ),
        "parameters": {"type": "object", "properties": {
            "지역": {"type": "string",
                    "description": "서울, 경기, 부산, 대구, 광주 중 하나. 예: '부산'. 생략하면 전체 지역"},
            "월": {"type": "integer",
                  "description": "1에서 12 사이의 정수. 예: 5. 생략하면 전체 월"}}},
    },
    {
        "name": "calculate",
        "description": (
            "이미 확보한 숫자들로 산술 연산을 수행합니다. 합계, 차이, 비율, 평균 계산에 사용하세요. "
            "데이터를 아직 조회하지 않았다면 먼저 query_sales를 사용하세요."
        ),
        "parameters": {"type": "object", "properties": {
            "expression": {"type": "string",
                           "description": "사칙연산 식만 허용. 예: '(1200+3400)/2'"}}},
    },
]

print("내 설명")
print(f"  정확도 {first_tool_accuracy(TOOLS_MY, tool_eval, show_miss=True):.0%}")

내 설명
  정확도 100%


In [21]:
PLANNER_SYSTEM = """당신은 데이터 분석 계획을 세우는 플래너입니다.
사용 가능한 도구는 다음과 같습니다.
- query_sales : 지역/월별 매출과 고객수 조회
- calculate : 산술 연산

사용자 질문을 해결하기 위한 단계를 JSON 배열로만 출력하세요. 설명을 덧붙이지 마세요.
action 값은 한국어로 작성하세요.
형식 : [{"step": 1, "action": "무엇을 할지", "tool": "도구명"}]"""


def make_plan(question):
    r = chat([{"role": "system", "content": PLANNER_SYSTEM},
              {"role": "user", "content": question}], max_new_tokens=400)
    text = re.sub(r"```(json)?", "", r["content"]).strip()
    m = re.search(r"\[.*\]", text, re.S)
    try:
        return json.loads(m.group(0)) if m else []
    except json.JSONDecodeError:
        return []


plan = make_plan("각 지역의 연간 총매출을 구하고, 가장 높은 지역과 가장 낮은 지역의 차이를 알려줘")
for p in plan:
    print(p)

{'step': 1, 'action': '연간 총매출을 계산하기 위해 각 지역의 매출을 합산', 'tool': 'query_sales'}
{'step': 2, 'action': '가장 높은 지역과 가장 낮은 지역을 찾기', 'tool': 'query_sales'}
{'step': 3, 'action': '가장 높은 지역과 가장 낮은 지역의 매출 차이를 계산', 'tool': 'calculate'}


In [22]:
# 계획을 실행하는 부분. 계획의 각 단계를 순서대로 수행합니다
def run_plan_and_act(question, verbose=True):
    plan = make_plan(question)
    if verbose:
        print("=== 계획 ===")
        for p in plan:
            print(f"  {p.get('step')}. {p.get('action')} ({p.get('tool')})")
        print("=== 실행 ===")

    messages = [{"role": "system", "content": AGENT_SYSTEM},
                {"role": "user", "content": question},
                {"role": "assistant", "content": f"수립한 계획: {json.dumps(plan, ensure_ascii=False)}"}]

    for step in range(len(plan) + 3):          # 계획보다 약간 여유를 둡니다
        r = chat(messages, tools=TOOLS)
        if not r["tool_calls"]:
            return {"answer": r["content"], "plan": plan}
        messages.append(assistant_tool_msg(r["content"], r["tool_calls"]))
        for c in r["tool_calls"]:
            out = safe_call(c["name"], c["arguments"])
            if verbose:
                print(f"  {c['name']}({c['arguments']})")
            messages.append({"role": "tool", "name": c["name"], "content": str(out)})

    return {"answer": "계획 실행이 완료되지 않았습니다.", "plan": plan}

In [26]:
r = run_plan_and_act("각 지역의 연간 총매출을 구하고, 가장 높은 지역과 가장 낮은 지역의 차이를 알려줘")
print("\n답변 :", r["answer"])

=== 계획 ===
  1. 연간 총매출을 계산하기 위해 각 지역의 매출을 합산합니다. (query_sales)
  2. 연간 총매출 중에서 가장 높은 지역과 가장 낮은 지역을 찾습니다. (query_sales)
  3. 가장 높은 지역과 가장 낮은 지역의 매출 차이를 계산합니다. (calculate)
=== 실행 ===
  query_sales({'지역': '서울'})
  query_sales({'지역': '경기'})
  query_sales({'지역': '부산'})
  query_sales({'지역': '대구'})
  query_sales({'지역': '광주'})

답변 : 다음으로, 각 지역의 연간 총매출을 계산하겠습니다. 이를 위해 매달의 매출을 모두 더해보겠습니다.

leton
{"name": "calculate", "arguments": {"expression": "(1660+3971+3191+3704+3547+3805+4356+1400+3624+1575+2385+3413)+(4572+3719+3233+1274+1775+2699+4690+3163+3412+4814+4743+4643)+(3892+930+1984+1882+2606+2067+1446+2861+2163+834+3873+2300)+(1266+2485+4185+3358+989+2328+3688+1041+2278+3952+1821+961)+(4244+1569+4917+2847+3534+4002+3235+2841+3356+2755+4261+2781}"}}


In [27]:
# 즉흥 루프와 비교
r2 = run_agent("각 지역의 연간 총매출을 구하고, 가장 높은 지역과 가장 낮은 지역의 차이를 알려줘",
               max_steps=8, verbose=False)
print(f"즉흥 루프 : {r2['steps']}단계")
print(f"답변 : {r2['answer'][:200]}")

즉흥 루프 : 8단계
답변 : 최대 단계에 도달했습니다.


In [28]:
#Step 4 : 로컬 RAG를 MCP 서버로 만들기
#검색 인덱스
%%writefile company_policy.md
# (주)아이펠테크 임직원 복무 규정

## 제2장 근무

### 제3조 (근무 시간)
기본 근무 시간은 주 40시간으로 하며, 1일 근무 시간은 휴게 시간을 제외하고 8시간으로 한다. 시업 시각은 오전 10시, 종업 시각은 오후 7시로 한다.

### 제5조 (재택근무)
임직원은 주 2회 이내에서 재택근무를 신청할 수 있다. 재택근무 신청은 근무일 전일 오후 6시까지 부서장의 승인을 받아야 한다.

## 제3장 휴가

### 제7조 (연차 휴가의 이월)
당해 연도에 사용하지 못한 연차 휴가는 다음 연도로 이월할 수 없음을 원칙으로 한다. 다만 회사의 업무상 사유로 사용하지 못한 경우에 한하여 인사팀장의 승인을 받아 최대 5일까지 다음 연도 1분기 내로 이월하여 사용할 수 있다.

### 제8조 (경조 휴가)
본인 결혼 5일, 자녀 결혼 1일, 배우자 출산 10일, 본인 및 배우자의 부모 사망 5일, 조부모 및 형제자매 사망 3일을 부여한다.

## 제4장 보안

### 제10조 (사내 데이터의 외부 반출)
고객 개인정보, 미공개 재무 정보, 기술 설계 문서를 포함한 사내 데이터는 정보보호팀장의 사전 승인 없이 외부로 반출할 수 없다. 외부 AI 서비스에 사내 문서를 입력하는 행위도 반출로 간주한다.

Writing company_policy.md


In [29]:
%%writefile policy_server.py
"""사내 규정 검색 MCP 서버.
로컬 임베딩 모델로 문서를 검색해 반환합니다. 외부 통신이 없습니다.
"""
from mcp.server.mcpserver import MCPServer
from mcp.types import ToolAnnotations
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
import numpy as np

mcp = MCPServer("policy")

# 서버 시작 시 인덱스를 만듭니다
with open("/content/company_policy.md", encoding="utf-8") as f:
    _doc = f.read()

_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[("##", "장"), ("###", "조항")]
)
# 조항이 없는 조각(문서 제목 줄)은 검색 대상에서 빼둡니다. 남겨두면 이름 없는
# 조각이 상위에 뽑혀 [] 만 붙은 결과가 나옵니다
_chunks = [c for c in _splitter.split_text(_doc) if c.metadata.get("조항")]
_emb = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-small",
    model_kwargs={"device": "cpu"},          # 서버는 CPU로 충분합니다
    encode_kwargs={"normalize_embeddings": True},
)
_vecs = np.array(_emb.embed_documents([f"passage: {c.page_content}" for c in _chunks]))


# 도구 주석. 이 두 도구는 읽기 전용이고 외부와 통신하지 않는다고 선언합니다.
# 달지 않으면 규격 기본값이 적용되어 "파괴적이고 외부와 통신함"으로 간주됩니다
_READ_ONLY = ToolAnnotations(read_only_hint=True, destructive_hint=False,
                             idempotent_hint=True, open_world_hint=False)


@mcp.tool(annotations=_READ_ONLY)
def search_policy(question: str, top_k: int = 2) -> str:
    """사내 규정 문서를 검색합니다.

    근무 시간, 재택근무, 연차와 경조 휴가, 데이터 보안 등 회사 내부 규칙에 관한
    질문일 때 사용하세요. 매출 데이터나 일반 상식 질문에는 사용하지 마세요.

    Args:
        question: 규정에 관한 자연어 질문. 예: '연차를 이월할 수 있나요?'
        top_k: 반환할 조항 개수. 기본 2개
    """
    q = np.array(_emb.embed_query(f"query: {question}"))
    order = np.argsort(_vecs @ q)[::-1][:top_k]
    out = []
    for i in order:
        c = _chunks[i]
        out.append(f"[{c.metadata.get('조항', '')}]\n{c.page_content}")
    return "\n\n".join(out)


@mcp.tool(annotations=_READ_ONLY)
def list_chapters() -> str:
    """규정 문서에 어떤 장이 있는지 목록을 반환합니다.

    사용자가 규정의 전체 구조를 알고 싶어 할 때, 또는 어느 장을 찾아야 할지
    불확실할 때 먼저 사용하세요.
    """
    seen = []
    for c in _chunks:
        ch = c.metadata.get("장", "")
        if ch and ch not in seen:
            seen.append(ch)
    return "\n".join(seen)


if __name__ == "__main__":
    mcp.run()

Writing policy_server.py


In [30]:
import sys
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# 주피터 계열 환경(실습 환경 포함)은 셀에서 최상위 await를 그대로 지원합니다.
# 예제에서 자주 보이는 nest_asyncio.apply()는 필요하지 않고,
# 이벤트 루프를 재진입 가능하게 만들어 오히려 문제를 일으킬 수 있어 쓰지 않습니다
SERVER_FILE = "/content/policy_server.py"

# command="python"이 아니라 sys.executable을 넘깁니다. 지금 이 커널을 돌리는 그 파이썬으로
# 서버를 띄우라는 뜻입니다. 실습 환경에서는 둘이 같지만, 가상환경을 쓰는 로컬에서는 PATH의
# python이 다른 인터프리터를 가리켜 서버가 ModuleNotFoundError로 죽습니다
server_params = StdioServerParameters(command=sys.executable, args=[SERVER_FILE])
SERVER_LOG = open("mcp_server.log", "a", encoding="utf-8")


async def mcp_list_tools():
    async with stdio_client(server_params, errlog=SERVER_LOG) as (r, w):
        async with ClientSession(r, w) as session:
            init = await session.initialize()          # 초기화 : 판본과 기능을 맞춥니다
            caps = init.capabilities
            print(f"합의된 프로토콜 판본 : {init.protocol_version}")
            print("서버가 선언한 기능   :",
                  [n for n in ("tools", "resources", "prompts", "logging", "completions")
                   if getattr(caps, n, None) is not None])
            print(f"자료 구독 지원       : {getattr(caps.resources, 'subscribe', None)}")
            print()
            tools = await session.list_tools()         # 운용 : 도구 목록을 받습니다
            return [(t.name, t.description, t.annotations) for t in tools.tools]


for name, desc, ann in await mcp_list_tools():
    print(f"[{name}]")
    print(f"  {desc.strip().splitlines()[0]}")
    if ann is None:
        print("  주석 없음 -> 규격 기본값 적용 (파괴적 O, 외부통신 O)")
    else:
        print(f"  읽기전용={ann.read_only_hint} 파괴적={ann.destructive_hint} "
              f"멱등={ann.idempotent_hint} 외부통신={ann.open_world_hint}")

합의된 프로토콜 판본 : 2025-11-25
서버가 선언한 기능   : ['tools', 'resources', 'prompts']
자료 구독 지원       : False

[search_policy]
  사내 규정 문서를 검색합니다.
  읽기전용=True 파괴적=False 멱등=True 외부통신=False
[list_chapters]
  규정 문서에 어떤 장이 있는지 목록을 반환합니다.
  읽기전용=True 파괴적=False 멱등=True 외부통신=False


In [31]:
# 운용 : 도구를 실제로 호출합니다
class MCPToolError(RuntimeError):
    """도구가 isError로 돌려준 실행 오류.

    retryable이 거짓이면 같은 인자로 다시 불러도 결과가 같습니다.
    """
    def __init__(self, text, retryable=True):
        super().__init__(text)
        self.retryable = retryable


def mcp_text(res):
    """호출 결과에서 본문을 꺼내고, 실행 오류면 예외로 올립니다"""
    text = "\n".join(c.text for c in res.content if hasattr(c, "text"))
    if res.is_error:
        # 인자가 스키마에 맞지 않는 실패는 재시도해도 같습니다.
        # 규격이 입력 검증 실패를 실행 오류로 다루라고 정한 덕에 여기서 구분할 수 있습니다
        raise MCPToolError(text, retryable="validation error" not in text)
    return text


async def mcp_call(tool_name, args):
    async with stdio_client(server_params, errlog=SERVER_LOG) as (r, w):
        async with ClientSession(r, w) as session:
            await session.initialize()
            res = await session.call_tool(tool_name, args)
            return mcp_text(res)


print(await mcp_call("search_policy", {"question": "연차를 다음 해로 넘길 수 있나요?"}))

[제7조 (연차 휴가의 이월)]
당해 연도에 사용하지 못한 연차 휴가는 다음 연도로 이월할 수 없음을 원칙으로 한다. 다만 회사의 업무상 사유로 사용하지 못한 경우에 한하여 인사팀장의 승인을 받아 최대 5일까지 다음 연도 1분기 내로 이월하여 사용할 수 있다.

[제5조 (재택근무)]
임직원은 주 2회 이내에서 재택근무를 신청할 수 있다. 재택근무 신청은 근무일 전일 오후 6시까지 부서장의 승인을 받아야 한다.


In [32]:
# 세션을 한 번만 열고 재사용하는 방식
import psutil
from contextlib import AsyncExitStack

_MCP = {"stack": None, "session": None}


async def mcp_session():
    """MCP 세션을 한 번만 열고 재사용합니다. 셀을 재실행해도 중복 기동하지 않습니다"""
    if _MCP["session"] is not None:
        return _MCP["session"]
    stack = AsyncExitStack()
    try:
        r, w = await stack.enter_async_context(stdio_client(server_params, errlog=SERVER_LOG))
        s = await stack.enter_async_context(ClientSession(r, w))
        await s.initialize()
    except BaseException:
        # 서버가 기동에 실패하면 반쯤 열린 스택을 여기서 반드시 닫습니다.
        # 남겨두면 다음 셀들이 전혀 다른 anyio 오류를 내며 원인을 가립니다
        try:
            await stack.aclose()
        except BaseException:
            pass
        raise
    _MCP["stack"], _MCP["session"] = stack, s
    print("MCP 세션을 새로 열었습니다")
    return s


async def mcp_call_fast(tool_name, args):
    """유지된 세션으로 도구를 호출합니다"""
    s = await mcp_session()
    res = await s.call_tool(tool_name, args)
    return mcp_text(res)


print(await mcp_call_fast("search_policy", {"question": "재택근무 신청 기한"}))

MCP 세션을 새로 열었습니다
[제5조 (재택근무)]
임직원은 주 2회 이내에서 재택근무를 신청할 수 있다. 재택근무 신청은 근무일 전일 오후 6시까지 부서장의 승인을 받아야 한다.

[제7조 (연차 휴가의 이월)]
당해 연도에 사용하지 못한 연차 휴가는 다음 연도로 이월할 수 없음을 원칙으로 한다. 다만 회사의 업무상 사유로 사용하지 못한 경우에 한하여 인사팀장의 승인을 받아 최대 5일까지 다음 연도 1분기 내로 이월하여 사용할 수 있다.


In [33]:
# 두 방식의 호출당 시간을 비교합니다
N = 3
q = {"question": "연차 이월"}

await mcp_call_fast("search_policy", q)     # 세션 개설 비용을 측정에서 빼기 위한 예비 호출

t0 = time.time()
for _ in range(N):
    await mcp_call_fast("search_policy", q)
fast = (time.time() - t0) / N

t0 = time.time()
for _ in range(N):
    await mcp_call("search_policy", q)
slow = (time.time() - t0) / N

print(f"세션 유지   : {fast*1000:>8.0f} ms/호출")
print(f"매번 재기동 : {slow*1000:>8.0f} ms/호출")
print(f"차이        : {slow/max(fast, 1e-6):>8.0f}배")

세션 유지   :       28 ms/호출
매번 재기동 :    21687 ms/호출
차이        :      765배


In [34]:
def _kill_own_servers():
    """이 노트북이 띄운 서버 자식 프로세스만 정리합니다 (2단계 안전망)"""
    killed = 0
    for ch in psutil.Process(os.getpid()).children(recursive=True):
        try:
            if any(SERVER_FILE in part for part in (ch.cmdline() or [])):
                ch.terminate(); killed += 1
        except psutil.Error:
            pass
    return killed


async def mcp_reset():
    """서버 파일을 고친 뒤 호출합니다. 다음 mcp_session()이 새 서버를 띄웁니다"""
    # 1단계 : 세션 닫기. 표준 입력이 닫히면 서버가 종료 신호로 읽습니다
    if _MCP["stack"] is not None:
        try:
            await _MCP["stack"].aclose()
        except BaseException:
            # 세션을 연 태스크와 닫는 태스크가 다르면 anyio가 여기서 예외를 냅니다.
            # 표준 입력은 이미 닫혔으므로 서버는 대개 정상 종료합니다
            pass
    # 2단계 : 참조 비우기
    _MCP["stack"], _MCP["session"] = None, None
    # 3단계 : 안전망. 1단계가 온전히 끝났으면 여기서 셀 것이 없습니다
    n = _kill_own_servers()
    print(f"세션을 닫았습니다 (안전망이 추가로 정리한 서버 {n}개. 0이면 정상 종료)")


print("mcp_reset 준비 완료")

mcp_reset 준비 완료


In [35]:
# MCP 서버의 도구 스키마를 에이전트가 쓸 형식으로 변환합니다
async def mcp_tool_schemas():
    async with stdio_client(server_params, errlog=SERVER_LOG) as (r, w):
        async with ClientSession(r, w) as session:
            await session.initialize()
            tools = await session.list_tools()
            return [{"name": t.name,
                     "description": t.description,
                     # 규격의 전선 형식은 inputSchema지만 파이썬 SDK 2.x는
                     # input_schema로 노출합니다. 주석 필드도 같은 규칙입니다
                     # (readOnlyHint -> read_only_hint)
                     "parameters": t.input_schema} for t in tools.tools]


MCP_TOOLS = await mcp_tool_schemas()
print([t["name"] for t in MCP_TOOLS])

['search_policy', 'list_chapters']


In [36]:
# 의도적으로 실패하는 도구를 추가합니다. 복구 로직을 시험하기 위한 장치입니다
FAIL_COUNT = {"n": 0}

def flaky_report(title: str) -> str:
    """보고서를 생성합니다. 사용자가 명시적으로 보고서 작성을 요청할 때만 사용하세요.

    Args:
        title: 보고서 제목
    """
    FAIL_COUNT["n"] += 1
    if FAIL_COUNT["n"] % 3 != 0:            # 세 번 중 두 번 실패
        raise RuntimeError("보고서 서버 일시 장애 (503)")
    return f"보고서 '{title}' 생성 완료"


TOOL_IMPL["flaky_report"] = flaky_report

FLAKY_SCHEMA = {
    "name": "flaky_report",
    "description": "보고서를 생성합니다. 사용자가 명시적으로 보고서 작성을 요청할 때만 사용하세요.",
    "parameters": {"type": "object",
                   "properties": {"title": {"type": "string", "description": "보고서 제목"}},
                   "required": ["title"]},
}

# append 대신 선언형으로 조립합니다.
# append를 쓰면 셀을 두 번 실행할 때 같은 도구가 두 번 들어가 프롬프트가 오염됩니다
ALL_TOOLS = TOOLS + MCP_TOOLS + [FLAKY_SCHEMA]
print([t["name"] for t in ALL_TOOLS])

['query_sales', 'calculate', 'search_policy', 'list_chapters', 'flaky_report']


In [37]:
# 실습 : 도구를 실행하는 함수에 복구 장치를 구현하세요
# 요구사항
#  1) MCP 도구면 mcp_call_fast로, 로컬 도구면 TOOL_IMPL로 실행
#  2) 일시 장애면 최대 3회까지 재시도
#  3) 인자가 틀린 실패는 재시도하지 말고 즉시 모델에게 되돌리기
#     (로컬 도구는 TypeError, MCP 도구는 MCPToolError의 retryable=False)
#  4) 마지막까지 실패하면 실패 사실을 문자열로 반환 (모델이 읽고 판단하게 함)

MCP_TOOL_NAMES = {t["name"] for t in MCP_TOOLS}

async def execute_tool(name, args, max_retry=3):
    for attempt in range(max_retry):
        try:
            # 여기에 실행 분기와 재시도 로직을 작성하세요
            pass
        except Exception as e:
            # 마지막 시도였다면 실패 메시지를 반환하세요
            pass
    return "도구 실행에 실패했습니다."


print(await execute_tool("flaky_report", {"title": "3월 매출 요약"}))

도구 실행에 실패했습니다.


In [38]:

MCP_TOOL_NAMES = {t["name"] for t in MCP_TOOLS}

async def execute_tool(name, args, max_retry=3):
    last_err = None
    for attempt in range(max_retry):
        try:
            if name in MCP_TOOL_NAMES:
                return await mcp_call_fast(name, args)      # 유지된 세션을 씁니다
            fn = TOOL_IMPL.get(name)
            if fn is None:
                return f"알 수 없는 도구입니다: {name}. 사용 가능한 도구만 호출하세요."
            return str(fn(**args))
        except MCPToolError as e:
            # MCP 도구의 실행 오류. 인자 문제면 재시도할 값어치가 없습니다
            if not e.retryable:
                return f"인자가 올바르지 않습니다: {e}. 도구 스키마를 확인해 다시 호출하세요."
            last_err = e
            if attempt < max_retry - 1:
                await asyncio.sleep(0.5 * (attempt + 1))
                continue
        except TypeError as e:
            # 로컬 도구의 인자 형식 오류. 같은 성질이므로 같이 다룹니다
            return f"인자가 올바르지 않습니다: {e}. 도구 스키마를 확인해 다시 호출하세요."
        except Exception as e:
            last_err = e
            if attempt < max_retry - 1:
                await asyncio.sleep(0.5 * (attempt + 1))     # 점진적 대기
                continue
    return f"도구 실행이 {max_retry}회 실패했습니다: {last_err}. 다른 방법을 시도하거나 실패를 보고하세요."

print(await execute_tool("flaky_report", {"title": "3월 매출 요약"}))

보고서 '3월 매출 요약' 생성 완료


In [39]:
# 루프 감지를 추가한 최종 에이전트
async def run_agent_v2(question, max_steps=8, verbose=True):
    messages = [{"role": "system", "content": AGENT_SYSTEM},
                {"role": "user", "content": question}]
    seen_calls = {}
    trace = []

    for step in range(max_steps):
        r = chat(messages, tools=ALL_TOOLS)

        if not r["tool_calls"]:
            return {"answer": r["content"], "trace": trace, "steps": step + 1}

        messages.append(assistant_tool_msg(r["content"], r["tool_calls"]))

        for c in r["tool_calls"]:
            key = (c["name"], json.dumps(c["arguments"], sort_keys=True, ensure_ascii=False))
            seen_calls[key] = seen_calls.get(key, 0) + 1

            if seen_calls[key] > 2:                      # 루프 감지
                out = ("동일한 호출이 반복되었습니다. 지금까지 얻은 정보로 "
                       "최종 답변을 작성하거나 다른 접근을 시도하세요.")
                if verbose:
                    print(f"[{step+1}] 루프 감지 : {c['name']}")
            else:
                out = await execute_tool(c["name"], c["arguments"])
                if verbose:
                    print(f"[{step+1}] {c['name']}({c['arguments']}) -> {str(out)[:70]}")

            trace.append({"step": step + 1, "tool": c["name"], "result": str(out)[:80]})
            messages.append({"role": "tool", "name": c["name"], "content": str(out)})

    return {"answer": "최대 단계에 도달했습니다.", "trace": trace, "steps": max_steps}

In [40]:
# MCP 도구를 쓰는 질문
r = await run_agent_v2("재택근무는 며칠 전까지 신청해야 해?")
print("\n답변 :", r["answer"])

[1] search_policy({'question': '재택근무는 며칠 전까지 신청해야 하나요?'}) -> [제5조 (재택근무)]
임직원은 주 2회 이내에서 재택근무를 신청할 수 있다. 재택근무 신청은 근무일 전일 오후 6시까지 부서

답변 : 재택근무는 근무일 전일 오후 6시까지 신청해야 합니다. 이를 통해 부서장의 승인을 받게 됩니다.


In [41]:
# 실패하는 도구를 쓰는 질문. 재시도가 동작하는지 확인합니다
FAIL_COUNT["n"] = 0
r = await run_agent_v2("'3월 매출 요약'이라는 제목으로 보고서를 만들어줘")
print("\n답변 :", r["answer"])

[1] flaky_report({'title': '3월 매출 요약'}) -> 보고서 '3월 매출 요약' 생성 완료

답변 : "3월 매출 요약"이라는 제목의 보고서를 성공적으로 생성하였습니다.


In [42]:
# 두 종류의 도구를 함께 써야 하는 질문
r = await run_agent_v2("연차 이월 규정을 알려주고, 서울 3월 매출도 함께 알려줘")
print("\n답변 :", r["answer"])

[1] search_policy({'question': '연차를 이월할 수 있나요?', 'top_k': 2}) -> [제7조 (연차 휴가의 이월)]
당해 연도에 사용하지 못한 연차 휴가는 다음 연도로 이월할 수 없음을 원칙으로 한다. 다만 회
[1] query_sales({'지역': '서울', '월': 3}) -> 지역  월   매출  고객수
서울  3 3191   80

답변 : 연차 이월 규정에 따르면, 당해 연도에 사용하지 못한 연차 휴가는 다음 연도로 이월할 수 없으며, 회사의 업무상 사유로 사용하지 못한 경우에는 인사팀장의 승인을 받아 최대 5일까지 다음 연도 1분기 내로 이월하여 사용할 수 있습니다.

서울 지역의 3월 매출은 3,191만원이며, 고객수는 80명입니다.
